In [1]:
import torch
from argparse import Namespace
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([
    Namespace,
    Phonemer_Tokenizer_Recombination,
    Series,
])


/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._r

In [2]:
import torch

# Monkey-patch torch.load to default to weights_only=False (Torch 2.6+ defaults to True)
_orig_torch_load = torch.load

def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_no_weights_only
print("✅ Patched torch.load to default weights_only=False for this kernel session.")


✅ Patched torch.load to default weights_only=False for this kernel session.


In [3]:
import wandb, random

run = wandb.init(
    entity="krishrawat0222-f",  # 👈 replace with your wandb username or team name
    project="DeepfakeDetectionRenewed",
    config={
        "test_run": True,
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "dummy",
        "epochs": 10,
    },
)

for epoch in range(10):
    acc = 1 - 2**-epoch - random.random() / epoch if epoch > 0 else 0.1
    loss = 2**-epoch + random.random() / (epoch + 1)
    run.log({"acc": acc, "loss": loss})

run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: yashaspatil (krishrawat0222-f) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


acc,▁▂▅▅▇▇▇███
loss,█▅▃▂▁▂▁▁▁▁
acc,0.98168
loss,0.02293


## Phoneme Recognition Model

In [4]:
import os, sys
# Add the directory containing the notebook to sys.path
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

In [5]:
from phoneme_GAT.phoneme_model import BaseModule, load_phoneme_model, optim_param

/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [6]:
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination

1. You can download the pretrained phoneme recognition model in [google drive](https://drive.google.com/file/d/1SbqynkUQxxlhazklZz9OgcVK7Fl2aT-z/view?usp=drive_link).
2. Change `pretrained_path` to you own custom path.
3. Remember to change the `pretrained_path` and `vocab_path` in the `load_phoneme_model` function of `phoneme_GAT.phoneme_model`.

In [7]:
network_param = Namespace(
    network_name="WavLM",
    pretrained_path="pretrained/best-epoch=42-val-per=0.407000.ckpt",
    pretrained_name="microsoft/wavlm-base",   # ✅ add this
    freeze=True,
    freeze_transformer=True,
    eos_token="</s>",
    bos_token="<s>",
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|",
    vocab_size=200,
)


To build the phoneme recognition model,
1. you must specify the pretrained_path!!!! Please download the provided pretrained phoneme model; or you can train yourself model through `train_phoneme_model.py`.
2. in the `load_phoneme_model` function, you have to change the correct `vocab_path`

In [8]:
from phoneme_GAT.phoneme_model import BaseModule, load_phoneme_model, optim_param

total_num_phonemes = 687  ## 198, or 687

phoneme_model = load_phoneme_model(
    network_name=network_param.network_name,
    pretrained_path=network_param.pretrained_path,
    total_num_phonemes=total_num_phonemes,
)
assert len(phoneme_model.tokenizer.total_phonemes) == total_num_phonemes

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


# Load model

## Audio model

In [9]:
from phoneme_GAT.modules import Phoneme_GAT_lit,Phoneme_GAT

In [10]:
audio_model = Phoneme_GAT(
    backbone='wavlm',
    use_raw=0,
    use_GAT=1,
    n_edges=10,
)

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


Generate a random audio to test the model.

## In-the-Wild evaluation dataset
This notebook mirrors `validation.ipynb`, but swaps the ASVspoof validation loader for the locally cached In-the-Wild dataset (`mueller91/In-The-Wild`).


In [11]:
import os

# ----------------------------
# Load HF token from secret.txt
# ----------------------------
def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path, "r") as f:
            return f.read().strip()
    return None

DATASET_NAME      = "mueller91/In-The-Wild"
CACHE_DIR         = "./data/in_the_wild"
SUBSET_SAMPLES    = None
LOCAL_FILES_ONLY  = True   # set False only if you need to re-download / refresh cache

HF_TOKEN = load_hf_token()

print("Dataset name     :", DATASET_NAME)
print("Cache dir        :", CACHE_DIR)
print("Subset samples   :", SUBSET_SAMPLES)
print("Local files only :", LOCAL_FILES_ONLY)
print("HF token present :", "✅ Yes" if HF_TOKEN else "❌ No (fine if dataset is already cached)")


Dataset name     : mueller91/In-The-Wild
Cache dir        : ./data/in_the_wild
Subset samples   : None
Local files only : True
HF token present : ✅ Yes


In [13]:
from datasets import load_dataset, Audio
from collections import Counter
import torch

print("=" * 80)
print("Loading In-the-Wild from cache and verifying decoding…")
print("=" * 80)

from datasets import DownloadConfig

download_config = DownloadConfig(local_files_only=LOCAL_FILES_ONLY) if LOCAL_FILES_ONLY else None

ds_itw = load_dataset(
    DATASET_NAME,
    cache_dir=CACHE_DIR,
    token=HF_TOKEN if HF_TOKEN else None,
    download_config=download_config,
)

print("Splits:", ds_itw)

ITW_SPLIT_NAME = "test" if "test" in ds_itw else list(ds_itw.keys())[0]
ds_itw_split = ds_itw[ITW_SPLIT_NAME].cast_column("audio", Audio(sampling_rate=16000))

if SUBSET_SAMPLES is None:
    subset_size = len(ds_itw_split)
else:
    subset_size = min(SUBSET_SAMPLES, len(ds_itw_split))

if subset_size < len(ds_itw_split):
    ds_itw_eval = ds_itw_split.shuffle(seed=42).select(range(subset_size))
else:
    ds_itw_eval = ds_itw_split

print(f"Using split       : {ITW_SPLIT_NAME}")
print(f"Full split size   : {len(ds_itw_split)}")
print(f"Eval subset size  : {len(ds_itw_eval)}")
print("Columns           :", ds_itw_eval.column_names)
print("Features          :", ds_itw_eval.features)

sample0 = ds_itw_eval[0]
print("\nFirst sample keys:", list(sample0.keys()))
audio0 = sample0["audio"]
print("Audio fields present:", list(audio0.keys()))
print("Audio sampling_rate :", audio0.get("sampling_rate"))
print(
    "Audio array dtype/len:",
    type(audio0.get("array")).__name__,
    len(audio0.get("array")) if audio0.get("array") is not None else None,
)

ITW_LABEL_COL = next(
    (k for k in sample0.keys() if "label" in k.lower() or k.lower() in ("key", "speaker", "type", "class")),
    None,
)
print("Detected label column:", ITW_LABEL_COL)

if ITW_LABEL_COL is None:
    raise ValueError(
        "Could not automatically detect the In-the-Wild label column. "
        "Inspect the printed keys above and set ITW_LABEL_COL manually."
    )

labels_preview = [ds_itw_eval[i][ITW_LABEL_COL] for i in range(min(200, len(ds_itw_eval)))]
print("Label sample (first 200):", Counter(labels_preview))


Loading In-the-Wild from cache and verifying decoding…
Splits: DatasetDict({
    train: Dataset({
        features: ['audio'],
        num_rows: 31779
    })
})
Using split       : train
Full split size   : 31779
Eval subset size  : 31779
Columns           : ['audio']
Features          : {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None)}

First sample keys: ['audio']
Audio fields present: ['path', 'array', 'sampling_rate']
Audio sampling_rate : 16000
Audio array dtype/len: ndarray 29105
Detected label column: None


ValueError: Could not automatically detect the In-the-Wild label column. Inspect the printed keys above and set ITW_LABEL_COL manually.

In [ ]:
from torch.utils.data import DataLoader
from loader import _label_to_int, _crop_policy, TARGET_SR

class ITWDataset(torch.utils.data.Dataset):
    """
    In-the-Wild dataset wrapper compatible with the existing validation pipeline.
    Expected labels: bonafide -> 0, spoof -> 1
    """
    def __init__(self, hf_split, label_col, mode="eval"):
        self.ds = hf_split
        self.label_col = label_col
        self.mode = mode

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        wav = _crop_policy(wav, self.mode)
        y = _label_to_int(ex[self.label_col])
        return {
            "audio": wav,
            "label": torch.tensor(y).long(),
            "sample_rate": TARGET_SR,
        }

itw_dataset = ITWDataset(ds_itw_eval, ITW_LABEL_COL, mode="eval")

val_dataloader = DataLoader(
    itw_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

print(f"In-the-Wild dataloader ready: {len(val_dataloader.dataset)} samples")


In [ ]:
import json
# ─────────────────────────────────────────────────────────────────────────────
# AUGMENTATION ATTACK SUITE  ·  Run ONE augmentation at a time + audition 10 clips
# ─────────────────────────────────────────────────────────────────────────────
import random, torch, numpy as np, math
import IPython.display as ipd
from IPython.display import display
import torchaudio
import torchaudio.functional as F
from fractions import Fraction

SAMPLE_RATE = 16_000
LISTEN_N    = 10
SEED        = 42
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

# ── ✏️  CHANGE THESE ─────────────────────────────────────────────────────────
ACTIVE_AUG = "reverberation"
# Options:
#   "pitch_up"        – pitch shift up N semitones
#   "pitch_down"      – pitch shift down N semitones
#   "phase_noise"     – small random phase perturbation
#   "amp_scale"       – fixed amplitude multiplier
#   "volume"          – gain in dB
#   "additive_noise"  – white noise at target SNR (dB)
#   "time_stretch"    – speed up/down without pitch change
#   "lowpass"         – low-pass filter at cutoff Hz
#   "highpass"        – high-pass filter at cutoff Hz
#   "reverberation"   – simulated room reverb
#   "codec_mulaw"     – mu-law 8-bit telephone codec

AUG_PARAMS = {
    "pitch_up":       {"semitones": 2},
    "pitch_down":     {"semitones": 2},
    "phase_noise":    {"noise_level": 0.012},
    "amp_scale":      {"scale": 1.20},
    "volume":         {"gain_db": 4.0},
    "additive_noise": {"snr_db": 32},
    "time_stretch":   {"rate": 1.07},
    "lowpass":        {"cutoff_hz": 6500},
    "highpass":       {"cutoff_hz": 120},
    "reverberation":  {"t60": 0.6, "room_scale": 0.5},
    "codec_mulaw":    {},
}
# ─────────────────────────────────────────────────────────────────────────────

def _match_length(x, target_len):
    if x.shape[-1] < target_len:
        x = torch.nn.functional.pad(x, (0, target_len - x.shape[-1]))
    return x[..., :target_len]

def aug_pitch_up(wav, semitones):
    factor = 2 ** (semitones / 12)
    frac   = Fraction(factor).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    return _match_length(stretched, wav.shape[-1])

def aug_pitch_down(wav, semitones):
    factor = 2 ** (-semitones / 12)
    frac   = Fraction(factor).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    return _match_length(stretched, wav.shape[-1])

def aug_phase_noise(wav, noise_level):
    spec  = torch.fft.rfft(wav)
    phase = torch.angle(spec)
    mag   = torch.abs(spec)
    phase = phase + torch.randn_like(phase) * noise_level * torch.pi
    return torch.fft.irfft(mag * torch.exp(1j * phase), n=wav.shape[-1])

def aug_amp_scale(wav, scale):
    return wav * scale

def aug_volume(wav, gain_db):
    gain = 10 ** (gain_db / 20)
    return wav * gain

def aug_additive_noise(wav, snr_db):
    sig_power   = wav.pow(2).mean()
    noise       = torch.randn_like(wav)
    noise_power = noise.pow(2).mean()
    scale       = (sig_power / (noise_power * 10 ** (snr_db / 10))).sqrt()
    return wav + scale * noise

def aug_time_stretch(wav, rate):
    orig_len = wav.shape[-1]
    frac     = Fraction(rate).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    return _match_length(stretched, orig_len)

def aug_lowpass(wav, cutoff_hz):
    return F.lowpass_biquad(wav, SAMPLE_RATE, cutoff_freq=cutoff_hz)

def aug_highpass(wav, cutoff_hz):
    return F.highpass_biquad(wav, SAMPLE_RATE, cutoff_freq=cutoff_hz)

def aug_reverberation(wav, t60=0.6, room_scale=0.5):
    B, C, T = wav.shape
    rir_len = min(int(t60 * SAMPLE_RATE), 1600)
    t       = torch.linspace(0, t60, rir_len, device=wav.device)
    decay   = torch.exp(-6.9 * t / t60)
    rir     = torch.randn(rir_len, device=wav.device) * decay

    for delay_ms in [15, 30, 50]:
        d = int(delay_ms * 1e-3 * SAMPLE_RATE * room_scale)
        if d < rir_len:
            rir[d] += 0.4 * room_scale * decay[d]

    rir   = rir / (rir.abs().max() + 1e-8)
    n_fft = 2 ** math.ceil(math.log2(T + rir_len - 1))
    wav_f = torch.fft.rfft(wav, n=n_fft)
    rir_f = torch.fft.rfft(rir, n=n_fft)
    out   = torch.fft.irfft(wav_f * rir_f, n=n_fft)
    return out[..., :T]

def aug_codec_mulaw(wav):
    wav8    = F.resample(wav, SAMPLE_RATE, 8000)
    encoded = torchaudio.functional.mu_law_encoding(wav8, quantization_channels=256)
    decoded = torchaudio.functional.mu_law_decoding(encoded, quantization_channels=256)
    wav16   = F.resample(decoded, 8000, SAMPLE_RATE)
    return _match_length(wav16, wav.shape[-1])

# ─────────────────────────────────────────────────────────────────────────────

AUGMENTATIONS = {
    "pitch_up":       lambda w: aug_pitch_up(w,        **AUG_PARAMS["pitch_up"]),
    "pitch_down":     lambda w: aug_pitch_down(w,      **AUG_PARAMS["pitch_down"]),
    "phase_noise":    lambda w: aug_phase_noise(w,     **AUG_PARAMS["phase_noise"]),
    "amp_scale":      lambda w: aug_amp_scale(w,       **AUG_PARAMS["amp_scale"]),
    "volume":         lambda w: aug_volume(w,          **AUG_PARAMS["volume"]),
    "additive_noise": lambda w: aug_additive_noise(w,  **AUG_PARAMS["additive_noise"]),
    "time_stretch":   lambda w: aug_time_stretch(w,    **AUG_PARAMS["time_stretch"]),
    "lowpass":        lambda w: aug_lowpass(w,         **AUG_PARAMS["lowpass"]),
    "highpass":       lambda w: aug_highpass(w,        **AUG_PARAMS["highpass"]),
    "reverberation":  lambda w: aug_reverberation(w,   **AUG_PARAMS["reverberation"]),
    "codec_mulaw":    lambda w: aug_codec_mulaw(w),
}

assert ACTIVE_AUG in AUGMENTATIONS, \
    f"Unknown augmentation '{ACTIVE_AUG}'. Choose from: {list(AUGMENTATIONS.keys())}"

aug_fn        = AUGMENTATIONS[ACTIVE_AUG]
active_params = AUG_PARAMS[ACTIVE_AUG]

# ── Run selected augmentation over full val dataloader ───────────────────────
chunks = []
print(f"Running augmentation: '{ACTIVE_AUG}' | params: {active_params}\n")

for batch_idx, batch in enumerate(val_dataloader):
    wav = batch["audio"]
    if wav.dim() == 2:
        wav = wav.unsqueeze(1)

    try:
        chunks.append(aug_fn(wav.clone()).cpu())
    except Exception as e:
        print(f"  [WARN] batch {batch_idx} failed: {e}")

    if (batch_idx + 1) % 10 == 0:
        print(f"  processed ~{(batch_idx + 1) * val_dataloader.batch_size} samples …")

# Concat
if not chunks:
    raise ValueError("No augmented batches were produced; check warnings above.")

max_len = max(c.shape[-1] for c in chunks)
aug_tensor = torch.cat(
    [torch.nn.functional.pad(c, (0, max_len - c.shape[-1])) for c in chunks], dim=0
)
print(f"\n✅  '{ACTIVE_AUG}' {active_params}  →  tensor shape: {aug_tensor.shape}")

# ── Audition 10 random clips ─────────────────────────────────────────────────
listen_idx = random.sample(range(aug_tensor.shape[0]), min(LISTEN_N, aug_tensor.shape[0]))
print(f"\n🎧  {len(listen_idx)} random clips  ──  {ACTIVE_AUG} {active_params}\n{'─'*50}")

for rank, idx in enumerate(listen_idx, 1):
    wav_np = aug_tensor[idx, 0].numpy().astype(np.float32)
    peak   = np.abs(wav_np).max()
    if peak > 0:
        wav_np /= peak
    print(f"  clip {rank:>2d}  (val idx {idx})")
    display(ipd.Audio(wav_np, rate=SAMPLE_RATE, normalize=False))

# Lit model

The settings of the `AudioModel` are defined in the `cfg`. Each setting is a key-value pair, where the key is the name of the setting and the value is the value of the setting. The meaning of each setting is defined as follows:

1. **Network Structure Parameters**:
   - `backbone` : "wavlm", the backbone of the phoneme recognition model.
   - `use_raw` : `False`, whether to use raw transformer as the backbone
   - `use_GAT`: `True`, whether to use GAT
   - `n_edges`: `10`, the nubmer of edges for each node in the GAT
   - `use_pool`: `True`, whether to use pooling


2. **Loss Function Parameters**:
   - `use_clip`: `True`, whether to use clip loss


3. **Data Augmentation and Training Strategy**:
   - `use_aug`: `True`, whether to use data augmentation in the training


In [ ]:
from argparse import Namespace

# Construct the configuration using Namespace
cfg = Namespace(
    PhonemeGAT=Namespace(
        backbone="wavlm",  # wavlm or wav2vec
        use_raw=False,              # whether to use raw transformer as the backbone
        use_GAT=True,              # whether to use GAT
        n_edges=10,                # the nubmer of edges for each node in the GAT
        use_aug=True,              # whether to use data augmentation in the training
        use_pool=True,            # whether to use pooling
        use_clip=True,             # whether to use clip loss
    )
)

In [ ]:
from pytorch_lightning import Trainer, LightningModule
from pytorch_lightning.loggers import  CSVLogger

We use the pytorch Lightning module to train the model, where we define the train step, validation/predict step, loss function and optimizer.

In [ ]:
audio_model_lit = Phoneme_GAT_lit(cfg=cfg)

## Test forwarding 

In the lit model, we use the `_shared_pred` method to predict the logits of the input batch. If the stage is train, we also the the audio_transform to augment the spectrogram.

Generate a random batch:

In [ ]:
x = torch.randn(3, 1, 48000)
batch = {
    "label": torch.randint(0, 2, (3,)),
    "audio": x,
    "sample_rate": 16000,
}

Note, you batch must be a dict with above keys.

In [ ]:
batch_res = audio_model_lit._shared_pred(batch=batch, batch_idx=0)
for key, value in batch_res.items():
    print(key, value.shape)

## Demo training

We first build a simple dataloaders for training, where all the samples are randomly generated.

In [ ]:
# from callbacks import EER_Callback, BinaryAUC_Callback, BinaryACC_Callback

# There code was balls so I rewrote it to be simple
from callbacks_rational import BinaryACC_Callback, BinaryAUC_Callback, EER_Callback, TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class SimpleTestDataset(Dataset):
    def __init__(self, num_samples=10):
        # Generate synthetic data similar to your example
        self.samples = []
        for _ in range(num_samples):
            self.samples.append({
                "audio": torch.randn(1, 48000),
                "label": torch.randint(0, 2, (1,)).item(),
                "sample_rate": 16000,
            })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]



# Create the dataset and dataloader
test_dataset = SimpleTestDataset(num_samples=20)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=3,
    shuffle=False,
)

We build a simple trainer to train and test our model.

In [ ]:
import wandb
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Finish any existing wandb run
if wandb.run is not None:
    wandb.finish()

wandb_logger = WandbLogger(
    project="DeepfakeDetectionRenewed",
    entity="krishrawat0222-f",
    name=f"attack_itw_{ACTIVE_AUG}",
    log_model=False,
    tags=["attack", "in-the-wild", ACTIVE_AUG],
)

trainer = Trainer(
    logger=wandb_logger,
    callbacks=[
        BinaryACC_Callback(batch_key="label", output_key="logit"),
        BinaryAUC_Callback(batch_key="label", output_key="logit"),
        EER_Callback(batch_key="label", output_key="logit"),
        TPR_Callback(batch_key="label", output_key="logit"),
        TNR_Callback(batch_key="label", output_key="logit"),
        FPR_Callback(batch_key="label", output_key="logit"),
        FNR_Callback(batch_key="label", output_key="logit"),
    ],
)

In [ ]:
CKPT_PATH = "robust_goat.ckpt"

model = Phoneme_GAT_lit.load_from_checkpoint(CKPT_PATH)
model = model.cuda()
torch.set_float32_matmul_precision('medium')
model.eval()
from torch.utils.data import DataLoader

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, aug_fn):
        self.base    = base_dataset
        self.aug_fn  = aug_fn

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = self.base[idx]
        wav  = item["audio"]                        # (1, T)
        wav  = self.aug_fn(wav.unsqueeze(0))        # add batch dim → (1,1,T)
        item["audio"] = wav.squeeze(0)              # back to (1, T)
        return item

# Wrap the existing In-the-Wild eval dataset
aug_val_dataset    = AugmentedDataset(val_dataloader.dataset, aug_fn)
aug_val_dataloader = DataLoader(
    aug_val_dataset,
    batch_size=val_dataloader.batch_size,
    shuffle=False,
    num_workers=val_dataloader.num_workers,
)

In [ ]:
print(next(model.parameters()).device)

In [ ]:


# Now validate on augmented inputs
trainer.validate(model=model, dataloaders=aug_val_dataloader)


After training, you can view the logging loss in the logger file, for example `logs/lightning_logs/version_0/metrics.csv`.
![](imgs/loss.png)

In [ ]:
import os
import csv
import json
import wandb
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger
from torch.utils.data import DataLoader

model = model.cuda()
torch.set_float32_matmul_precision("medium")

# ─────────────────────────────────────────────────────────────────────────────
# 1) 5 values per augmentation
# ─────────────────────────────────────────────────────────────────────────────
ATTACK_SWEEPS = {
    "pitch_up":       [{"semitones": v} for v in [1, 2, 3, 4, 5]],
    "pitch_down":     [{"semitones": v} for v in [1, 2, 3, 4, 5]],
    "phase_noise":    [{"noise_level": v} for v in [0.003, 0.006, 0.012, 0.020, 0.030]],
    "amp_scale":      [{"scale": v} for v in [0.80, 0.90, 1.00, 1.10, 1.20]],
    "volume":         [{"gain_db": v} for v in [-6.0, -3.0, 0.0, 3.0, 6.0]],
    "additive_noise": [{"snr_db": v} for v in [40, 32, 24, 16, 8]],
    "time_stretch":   [{"rate": v} for v in [0.75, 0.9, 1.07, 1.25, 1.5]],
    "lowpass":        [{"cutoff_hz": v} for v in [7000, 5000, 3500, 2500, 1500]],
    "highpass":       [{"cutoff_hz": v} for v in [40, 80, 120, 200, 350]],
    "reverberation":  [
        {"t60": 0.20, "room_scale": 0.25},
        {"t60": 0.35, "room_scale": 0.40},
        {"t60": 0.50, "room_scale": 0.50},
        {"t60": 0.70, "room_scale": 0.65},
        {"t60": 0.90, "room_scale": 0.80},
    ],
    "codec_mulaw":    [{}],   # deterministic, so 1 run is enough
}

SKIP_AUGS = set({"pitch_down","volume","lowpass","highpass","time_stretch","amp_scale"})
RESULTS_CSV = "augmentation_attack_results_itw.csv"

# ─────────────────────────────────────────────────────────────────────────────
# 2) Dynamic augmentation builder
# ─────────────────────────────────────────────────────────────────────────────
def build_aug_fn(aug_name, params):
    if aug_name == "pitch_up":
        return lambda w: aug_pitch_up(w, **params)
    elif aug_name == "pitch_down":
        return lambda w: aug_pitch_down(w, **params)
    elif aug_name == "phase_noise":
        return lambda w: aug_phase_noise(w, **params)
    elif aug_name == "amp_scale":
        return lambda w: aug_amp_scale(w, **params)
    elif aug_name == "volume":
        return lambda w: aug_volume(w, **params)
    elif aug_name == "additive_noise":
        return lambda w: aug_additive_noise(w, **params)
    elif aug_name == "time_stretch":
        return lambda w: aug_time_stretch(w, **params)
    elif aug_name == "lowpass":
        return lambda w: aug_lowpass(w, **params)
    elif aug_name == "highpass":
        return lambda w: aug_highpass(w, **params)
    elif aug_name == "reverberation":
        return lambda w: aug_reverberation(w, **params)
    elif aug_name == "codec_mulaw":
        return lambda w: aug_codec_mulaw(w)
    else:
        raise ValueError(f"Unknown augmentation: {aug_name}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) CSV helpers
# ─────────────────────────────────────────────────────────────────────────────
CSV_COLUMNS = [
    "aug_name",
    "sweep_idx",
    "param_json",
    "status",
    "error",
    "val_acc",
    "val_auc",
    "val_eer",
    "val_tpr",
    "val_tnr",
    "val_fpr",
    "val_fnr",
]

def append_result_to_csv(csv_path, row_dict):
    file_exists = os.path.exists(csv_path)

    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)
        f.flush()

def load_completed_runs(csv_path):
    completed = set()
    if not os.path.exists(csv_path):
        return completed

    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            completed.add((row["aug_name"], row["param_json"]))
    return completed

completed_runs = load_completed_runs(RESULTS_CSV)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Metric extraction helper
#    Adjust aliases if your callback names differ in Lightning output
# ─────────────────────────────────────────────────────────────────────────────
def normalize_metrics(metrics):
    out = {}

    aliases = {
        "val_acc": ["val_acc", "ACC", "acc", "BinaryACC"],
        "val_auc": ["val_auc", "AUC", "auc", "BinaryAUC"],
        "val_eer": ["val_eer", "EER", "eer"],
        "val_tpr": ["val_tpr", "TPR", "tpr"],
        "val_tnr": ["val_tnr", "TNR", "tnr"],
        "val_fpr": ["val_fpr", "FPR", "fpr"],
        "val_fnr": ["val_fnr", "FNR", "fnr"],
    }

    for canonical_name, candidates in aliases.items():
        value = None
        for c in candidates:
            if c in metrics:
                value = metrics[c]
                break
        out[canonical_name] = value

    return out

# ─────────────────────────────────────────────────────────────────────────────
# 5) Main sweep loop
# ─────────────────────────────────────────────────────────────────────────────
for aug_name, param_list in ATTACK_SWEEPS.items():
    if aug_name in SKIP_AUGS:
        print(f"[SKIP] {aug_name}")
        continue

    for sweep_idx, params in enumerate(param_list, start=1):
        param_json = json.dumps(params, sort_keys=True)

        print(f"\n{'='*70}")
        print(f"Running attack: {aug_name} | sweep {sweep_idx}/{len(param_list)} | params: {params}")
        print(f"{'='*70}")

        aug_fn_i = build_aug_fn(aug_name, params)

        aug_val_dataset = AugmentedDataset(val_dataloader.dataset, aug_fn_i)
        aug_val_dataloader = DataLoader(
            aug_val_dataset,
            batch_size=val_dataloader.batch_size,
            shuffle=False,
            num_workers=0,
        )

        if wandb.run is not None:
            wandb.finish()

        run_name = f"attack_itw_{aug_name}_{sweep_idx}"

        wandb_logger = WandbLogger(
            project="DeepfakeDetectionRenewed",
            entity="krishrawat0222-f",
            name=run_name,
            log_model=False,
            tags=["attack", "in-the-wild", aug_name, f"sweep_{sweep_idx}"],
        )

        # Log attack metadata into W&B config
        wandb_logger.experiment.config.update({
            "attack_family": aug_name,
            "attack_params": params,
            "attack_param_json": param_json,
            "sweep_idx": sweep_idx,
            "results_csv": RESULTS_CSV,
        }, allow_val_change=True)

        trainer = Trainer(
            accelerator="gpu",
            devices=1,
            logger=wandb_logger,
            callbacks=[
                BinaryACC_Callback(batch_key="label", output_key="logit"),
                BinaryAUC_Callback(batch_key="label", output_key="logit"),
                EER_Callback(batch_key="label", output_key="logit"),
                TPR_Callback(batch_key="label", output_key="logit"),
                TNR_Callback(batch_key="label", output_key="logit"),
                FPR_Callback(batch_key="label", output_key="logit"),
                FNR_Callback(batch_key="label", output_key="logit"),
            ],
        )

        try:
            val_results = trainer.validate(model=model, dataloaders=aug_val_dataloader)
            raw_metrics = val_results[0] if isinstance(val_results, list) and len(val_results) > 0 else {}
            norm_metrics = normalize_metrics(raw_metrics)

            status = "ok"
            error_msg = ""

            # Explicitly log clean metrics to W&B too
            wandb_logger.experiment.log({
                "attack/aug_name": aug_name,
                "attack/sweep_idx": sweep_idx,
                **{f"attack_param/{k}": v for k, v in params.items()},
                **{f"metrics/{k}": v for k, v in norm_metrics.items() if v is not None},
            })

        except Exception as e:
            print(f"[WARN] {aug_name} with params {params} failed: {e}")
            raw_metrics = {}
            norm_metrics = {
                "val_acc": None,
                "val_auc": None,
                "val_eer": None,
                "val_tpr": None,
                "val_tnr": None,
                "val_fpr": None,
                "val_fnr": None,
            }
            status = "failed"
            error_msg = str(e)

            wandb_logger.experiment.log({
                "attack/aug_name": aug_name,
                "attack/sweep_idx": sweep_idx,
                "attack/status_failed": 1,
            })

        row = {
            "aug_name": aug_name,
            "sweep_idx": sweep_idx,
            "param_json": param_json,
            "status": status,
            "error": error_msg,
            **norm_metrics,
        }

        append_result_to_csv(RESULTS_CSV, row)
        completed_runs.add((aug_name, param_json))
        print(f"[CSV UPDATED] {RESULTS_CSV}")

        # Optional: upload CSV snapshot to the current W&B run
        if os.path.exists(RESULTS_CSV):
            wandb_logger.experiment.save(RESULTS_CSV, policy="now")

        if wandb.run is not None:
            wandb.finish()

print(f"\n✅ All augmentation sweeps complete. Results saved to: {RESULTS_CSV}")